In [1]:
import json
import re
import time
from datetime import datetime, timezone

import requests
from bs4 import BeautifulSoup

UA = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/122.0 Safari/537.36"
)

headers = {"User-Agent": UA, "Accept-Language": "es-AR,es;q=0.9,en;q=0.8"}

session = requests.Session()

def fetch(url: str) -> str:
    r = session.get(url, headers=headers, timeout=30)
    r.raise_for_status()
    return r.text

In [2]:
search_url = "https://inmuebles.mercadolibre.com.ar/venta-casa-caseros_NoIndex_True"
html = fetch(search_url)
len(html), html[:200]

(1537178,
 '<!DOCTYPE html><html lang="es-AR"><head><meta charSet="utf-8"/><link rel="preconnect" href="https://http2.mlstatic.com"/><meta name="viewport" content="width=device-width, initial-scale=1.0, maximum-s')

In [3]:
soup = BeautifulSoup(html, "lxml")

urls = []
for a in soup.select("a.poly-component__title"):
    href = a.get("href")
    if href:
        urls.append(str(href).split("#", 1)[0])

# dedupe preservando orden
seen = set()
urls_unique = []
for u in urls:
    if u not in seen:
        urls_unique.append(u)
        seen.add(u)

len(urls_unique), urls_unique[:5]

(48,
 ['https://casa.mercadolibre.com.ar/MLA-1682911893-casa-en-venta-5-ambientes-caseros-3-de-febrero-_JM',
  'https://casa.mercadolibre.com.ar/MLA-2182161214-casa-4-dormitorios-de-generos-tamano-jardin-y-terraza-_JM',
  'https://casa.mercadolibre.com.ar/MLA-2065198804-casa-en-venta-en-caseros-norte-con-quincho-y-terraza-_JM',
  'https://casa.mercadolibre.com.ar/MLA-2866436422-casa-4-ambientes-con-jardin-y-apta-a-credito-en-venta-en-caseros-_JM',
  'https://inmueble.mercadolibre.com.ar/MLA-1678353251-venta-galpon-2-banos-casa-caseros-tres-de-febrero-_JM'])

In [4]:
item_url = urls_unique[0]
item_html = fetch(item_url)

item_soup = BeautifulSoup(item_html, "lxml")
scripts = item_soup.select('script[type="application/ld+json"]')

ld_list = []
for s in scripts:
    txt = s.string or s.get_text(strip=True)
    if not txt:
        continue
    ld_list.append(json.loads(txt))

len(ld_list), [obj.get("@type") for obj in ld_list]

(3, ['Product', 'BreadcrumbList', 'Table'])

In [ ]:
MLA_RE = re.compile(r"(MLA[-]?\d+)", re.IGNORECASE)

def extract_item_id(url: str, ld: list[dict]):
    for obj in ld:
        sku = obj.get("sku") or obj.get("productID")
        if isinstance(sku, str) and sku:
            return sku
    m = MLA_RE.search(url)
    return m.group(1).replace("-", "").upper() if m else None

def extract_price_currency(ld: list[dict]):
    for obj in ld:
        offers = obj.get("offers")
        if isinstance(offers, dict):
            return offers.get("price"), offers.get("priceCurrency")
    return None, None

item_id = extract_item_id(item_url, ld_list)
price, currency = extract_price_currency(ld_list)

item_url, item_id, price, currency

('https://casa.mercadolibre.com.ar/MLA-1682911893-casa-en-venta-5-ambientes-caseros-3-de-febrero-_JM',
 'MLA1682911893',
 99900,
 'USD')

In [6]:
def search_url_from_offset(offset: int) -> str:
    base = "https://inmuebles.mercadolibre.com.ar/venta-casa-caseros_NoIndex_True"
    if offset == 1:
        return base
    return base.replace("_NoIndex_True", f"_Desde_{offset}_NoIndex_True")

for offset in [1, 49, 97]:
    url = search_url_from_offset(offset)
    h = fetch(url)
    sp = BeautifulSoup(h, "lxml")
    n = len(sp.select("a.poly-component__title"))
    print(offset, url, "cards:", n)
    time.sleep(1)

1 https://inmuebles.mercadolibre.com.ar/venta-casa-caseros_NoIndex_True cards: 48
49 https://inmuebles.mercadolibre.com.ar/venta-casa-caseros_Desde_49_NoIndex_True cards: 48
97 https://inmuebles.mercadolibre.com.ar/venta-casa-caseros_Desde_97_NoIndex_True cards: 48
